# Script for endpoint fluorescence data analysis

## Import dependencies

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

In [ ]:
def combine(list1, list2, sep=""):
    new = []
    for el in list1:
        for el2 in list2:
            new.append(el+sep+el2)
    return(new)

## Fetch data and define wells/samples

In [ ]:
raw_od =  pd.read_excel("Terminator_Strength-250425-4h.xlsx", sheet_name="OD600")
raw_gfp =  pd.read_excel("Terminator_Strength-250425-4h.xlsx", sheet_name="GFP")
raw_rfp =  pd.read_excel("Terminator_Strength-250425-4h.xlsx", sheet_name="RFP")

In [ ]:
letters = ["A", "B", "C"]
numbers = [str(num+1) for num in range(9)]
sample_wells = combine(letters, numbers)

blank_wells = ["G11", "G12", "H11", "H12"]
af_wells = ["D1", "D2", "D3"]
samples = combine(["sfGFP_RNaseE", "sfGFP", "noTx_sfGFP"], ["J23105", "T7hyb1", "20bp"], sep="\n")

## Define functions for normalization

In [ ]:
def raw_stats(df):
    mean_df = pd.DataFrame()
    mean_df.index = df.index
    mean_df["Blank"] = df[blank_wells].mean(axis=1)
    mean_df["AF"] = df[af_wells].mean(axis=1)
    for idx, el in enumerate(samples):
        mean_df[el] = df[sample_wells[idx*3:idx*3+3]].mean(axis=1)
    
    
    std_df = pd.DataFrame()
    std_df.index = df.index
    std_df["Blank"] = df[blank_wells].std(axis=1)
    std_df["AF"] = df[af_wells].std(axis=1)
    for idx, el in enumerate(samples):
        std_df[el] = df[sample_wells[idx*3:idx*3+3]].std(axis=1)
        #print(el, "->", sample_wells[idx*3:idx*3+3])

    return mean_df, std_df

In [ ]:
def blank(df, std = False):
    blanked_df = pd.DataFrame()
    blanked_df.index = df.index
    if not std:
        blanked_df["AF"] = df["AF"] - df["Blank"]
        blanked_df[samples] = df[samples].sub(df["Blank"], axis=0)
    else:
        blanked_df["AF"] = np.sqrt(df["AF"]**2 + df["Blank"]**2)
        blanked_df[samples] = np.sqrt(np.power(df[samples], 2).add(np.power(df["Blank"], 2), axis=0))

    return blanked_df

In [ ]:
def remove_af(df, std = False):
    blanked_df = pd.DataFrame()
    blanked_df.index = df.index
    if not std:
        blanked_df[samples] = df[samples].sub(df["AF"], axis=0)
    else:
        blanked_df[samples] = np.sqrt(np.power(df[samples], 2).add(np.power(df["AF"], 2), axis=0))

    return blanked_df

## Calculate mean/std, divide by OD, remove AF

In [ ]:
mean_od, std_od = raw_stats(raw_od)
mean_gfp, std_gfp = raw_stats(raw_gfp)
mean_rfp, std_rfp = raw_stats(raw_rfp)

In [ ]:
blanked_mean_od = blank(mean_od)
blanked_std_od = blank(std_od, std=True)

blanked_mean_gfp = blank(mean_gfp)
blanked_std_gfp = blank(std_gfp, std=True)

blanked_mean_rfp = blank(mean_rfp)
blanked_std_rfp = blank(std_rfp, std=True)

In [ ]:
mean_G_OD = blanked_mean_gfp/blanked_mean_od
std_G_OD = abs(mean_G_OD) * np.sqrt((blanked_std_gfp/blanked_mean_gfp)**2 + (blanked_std_od/blanked_mean_od)**2)

mean_R_OD = blanked_mean_rfp/blanked_mean_od
std_R_OD = abs(mean_R_OD) * np.sqrt((blanked_std_rfp/blanked_mean_rfp)**2 + (blanked_std_od/blanked_mean_od)**2)

In [ ]:
mean_G_OD_AF = remove_af(mean_G_OD)
std_G_OD_AF = remove_af(std_G_OD, std=True)

mean_R_OD_AF = remove_af(mean_R_OD)
std_R_OD_AF = remove_af(std_R_OD, std=True)

## Plot with labels

In [ ]:
labels = mean_G_OD.columns
fig, axs = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)

#fig.suptitle("Uninduced plasmids in DH10B")


axs[0].grid(axis='y')
axs[0].set_axisbelow(True)
axs[0].bar(labels, mean_G_OD.loc[0], color="green")
axs[0].errorbar(labels, mean_G_OD.loc[0], yerr = std_G_OD.loc[0], marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
axs[0].set_xticks([0, 2, 5, 8])
axs[0].set_xticklabels(["AF", "sfGFP\nRNaseE", "sfGFP", "noTx\nsfGFP"], ha="center", rotation_mode="anchor")
axs[0].set_yscale('symlog')
axs[0].set_ylim(10**5, 10**6)
axs[0].set_yticks(np.linspace(10**5, 10**6, 10))
axs[0].set_ylabel("sfGFP/OD600 (a.u.)")
axs[0].tick_params(axis='x', length=0)

lines = [Line2D([0.5,0.5], [10**5, 7.5*10**4], lw=1, color="k", linestyle="--"),
         Line2D([3.5,3.5], [10**5, 7.5*10**4], lw=1, color="k", linestyle="--"), 
         Line2D([6.5,6.5], [10**5, 7.5*10**4], lw=1, color="k", linestyle="--")]
for line in lines:
    line.set_clip_on(False)
    axs[0].add_line(line)

for idx, sample in enumerate(samples):
    axs[0].text(idx + 1, 1.07*10**5, "+ " + sample.split("\n")[1], rotation=90, ha = "center", color="white")
    
axs[1].grid(axis='y')
axs[1].set_axisbelow(True)
axs[1].bar(labels, mean_R_OD.loc[0], color="red")
axs[1].errorbar(labels, mean_R_OD.loc[0], yerr = std_R_OD.loc[0], marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
axs[1].set_xticks([0, 2, 5, 8])
axs[1].set_xticklabels(["AF", "sfGFP\nRNaseE", "sfGFP", "noTx\nsfGFP"], ha="center", rotation_mode="anchor")
axs[1].set_yscale('symlog')
axs[1].set_ylim(10**2, 10**8)
axs[1].set_ylabel("mScarlet/OD600 (a.u.)")
axs[1].tick_params(axis='x', length=0)

lines = [Line2D([0.5,0.5], [10**2, 1.9*10**1], lw=1, color="k", linestyle="--"),
         Line2D([3.5,3.5], [10**2, 1.9*10**1], lw=1, color="k", linestyle="--"), 
         Line2D([6.5,6.5], [10**2, 1.9*10**1], lw=1, color="k", linestyle="--")]
for line in lines:
    line.set_clip_on(False)
    axs[1].add_line(line)
    
for idx, sample in enumerate(samples):
    axs[1].text(idx + 1, 1.5*10**2, "+ " + sample.split("\n")[1], rotation=90, ha = "center")

plt.savefig("Basal_Fluorescence.png")
plt.show()